# Ch.3 — Unsupervised Metrics

> **The story.** The hardest question in machine learning is not "how do I build a model?" but "how do I know if it worked?" In supervised learning the answer is straightforward: compare predictions to labels. In unsupervised learning the answer took decades to develop. **William M. Rand** proposed the Rand Index in **1971**, correcting for chance agreement between two clusterings. **Calinski and Harabasz** proposed their variance-ratio index in **1974**: between-cluster dispersion divided by within-cluster dispersion. **Davies and Bouldin** followed in **1979** with their per-cluster compactness ratio. The field crystallised with **Peter Rousseeuw**'s silhouette coefficient in **1987** — the per-point measure that asks _"is point $i$ closer to its own cluster or to the nearest other cluster?"_ — yielding a score in $[-1,1]$ any practitioner can interpret without consulting a statistician. Together these four milestones turned unsupervised learning from "pretty plots" into engineering decisions backed by quantitative evidence.
>
> **Where you are in the curriculum.** This is the **final chapter** of the Unsupervised Learning track. [Ch.1](../ch01_clustering) ran K-Means on UCI Wholesale customers and produced k=4 segments (silhouette=0.42 — below the 0.5 target). [Ch.2](../ch02_dimensionality_reduction) applied UMAP 3D to compress the feature space and re-ran K-Means, visually tightening the clusters. Now the CMO asks the hard engineering question: _"Are those 4 segments provably good?"_ This chapter provides the answer — four formal metrics that validate cluster quality without labels — and closes the SegmentAI mission with silhouette=0.57.
>
> **Notation in this chapter.** Clustering of $n$ points into $k$ clusters: $C_i$ — set of points in cluster $i$; $n_i=|C_i|$ — cluster size; $\mu_i$ — centroid of $C_i$; $\bar{\mu}$ — overall data centroid. **Silhouette:** $a(i)$ — mean distance from point $i$ to all other members of its own cluster; $b(i)$ — mean distance from $i$ to all members of its nearest other cluster; $s(i)=\frac{b(i)-a(i)}{\max(a(i),b(i))}\in[-1,1]$. **Davies–Bouldin:** $\sigma_i$ — mean intra-cluster distance; $\mathrm{DB}=\frac{1}{k}\sum_{i=1}^{k}\max_{j\neq i}\frac{\sigma_i+\sigma_j}{d(\mu_i,\mu_j)}$ — lower is better. **Calinski–Harabasz:** $B=\sum_i n_i\|\mu_i-\bar{\mu}\|^2$ — between-cluster SS; $W=\sum_i\sum_{x\in C_i}\|x-\mu_i\|^2$ — within-cluster SS; $\mathrm{CH}=\frac{B/(k-1)}{W/(n-k)}$ — higher is better.

---

## 0 · The Challenge

> **The mission**: Build **SegmentAI** — discover 4 actionable customer segments from UCI Wholesale data satisfying 5 constraints.

**What we know so far:**

- Ch.1: K-Means on 440 wholesale customers → k=4 segments, silhouette=0.42 (below 0.5 target)
- Ch.2: UMAP 3D compression → re-clustered → visually tighter clusters, silhouette improves
- **We have no formal proof that k=4 is optimal or that the clusters are not artefacts of random initialisation**

**What's blocking us:**

The CMO asks: _"Our marketing team is about to build four separate campaigns. How do we know those clusters are not noise?"_ In supervised learning there is always a right answer. In unsupervised learning there is no right answer — nobody labelled 440 wholesale customers as "HoReCa buyer." K-Means found 4 groups — but was the grouping _good_?

**What this chapter unlocks:**

The four canonical metrics for measuring cluster quality without labels: silhouette (cohesion vs separation), Davies–Bouldin (compactness ratio), Calinski–Harabasz (global variance ratio), and ARI (when proxy labels exist).

```mermaid
flowchart LR
 A["Ch.1: K-Means k=4\nsilhouette=0.42\nbelow 0.5 target"] --> B["Ch.2: UMAP 3D\ntighter clusters\nsilhouette improves"]
 B --> C["Ch.3: Metrics suite\nsilhouette=0.57\nALL 5 constraints met"]
 style A fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
 style C fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

| Supervised evaluation                 | Unsupervised evaluation    |
| ------------------------------------- | -------------------------- |
| Compare $\hat{y}$ to ground-truth $y$ | No $y$ available           |
| MAE, F1, AUC, R²                      | Silhouette, DB, CH, ARI    |
| One correct answer per prediction     | Geometric quality measures |
| Direct falsifiability                 | Requires metric agreement  |


## Core Idea

**Silhouette score:** For each customer, ask two questions: "How close am I to my own cluster-mates?" (cohesion, $a$) and "How far am I from the nearest other cluster?" (separation, $b$). A good assignment means $b \gg a$ — you're tightly grouped with similar customers and clearly separated from different ones.

> **Optional depth:** $s(i)=\frac{b(i)-a(i)}{\max(a(i),b(i))}\in[-1,1]$. When $b \gg a$: $s(i) \to 1$ (well placed). When $a \approx b$: $s(i) \approx 0$ (on a boundary). When $a > b$: $s(i) < 0$ (likely misassigned to the wrong cluster).

**Davies–Bouldin index:** For each cluster, measure how internally loose it is relative to how far it sits from its nearest neighbouring cluster. A cluster that is both loose internally _and_ close to another cluster is the worst possible configuration. DB averages the worst-case compactness-to-separation ratio over all clusters. Lower is better.

> **Optional depth:** $\mathrm{DB}=\frac{1}{k}\sum_{i=1}^{k}\max_{j\neq i}\frac{\sigma_i+\sigma_j}{d(\mu_i,\mu_j)}$ where $\sigma_i$ is the mean intra-cluster distance for cluster $i$ and $d(\mu_i,\mu_j)$ is the centroid distance. Perfect clusters give DB → 0.

**Calinski–Harabasz score:** Decompose total variance into "between clusters" ($B$ — how far apart are the centroids?) and "within clusters" ($W$ — how spread are points inside each cluster?). A high ratio means centroids are far apart relative to internal spread. Higher is better.

> **Optional depth:** $\mathrm{CH}=\frac{B/(k-1)}{W/(n-k)}$. CH is unbounded and grows with $n$, so only compare different $k$ values on the same dataset.

**Adjusted Rand Index:** When a proxy ground-truth label exists (like the `Channel` column — Hotel vs Retail), measure how well your clusters agree with it, corrected for random chance. ARI≈0 means no agreement above chance; ARI=1 means perfect agreement.

The key insight: require silhouette, DB, and CH to _agree_ on a k value. When they agree, the evidence is geometric and robust. When they disagree, the data has ambiguous structure and the business requirement should break the tie.


## How Silhouette Works — The Geometry

```
Silhouette coefficient — per-point view:

Cluster A (tight)        Cluster B (loose)
  ○ ○                         ○  ○
○ i ○  ← point i          ○    ○
  ○ ○                       ○  ○

  a(i) = mean distance from i to its own cluster-mates (cohesion)
         Small a(i) → tight cluster — good

  b(i) = mean distance from i to nearest OTHER cluster
         Large b(i) → well separated — good

  s(i) = (b(i) - a(i)) / max(a(i), b(i))

  s(i) → +1: i is well inside its cluster, far from others → perfect
  s(i) → 0:  i is on the boundary between two clusters
  s(i) → -1: i is closer to a different cluster → likely misassigned
```

The mean silhouette across all points is the overall score. But the mean can hide a badly-placed cluster. The silhouette subplot (one bar per cluster) reveals which segments are well-defined and which are borderline.


## Running Example — SegmentAI

The CMO is one meeting away from approving four separate marketing campaigns — one per customer segment. Before that meeting, the data team needs to answer one question: "Are these four segments real, or did K-Means just draw arbitrary lines?" There are no ground-truth labels to check against. The only evidence available is the geometry of the clusters themselves. That is what unsupervised metrics measure.

Dataset: **UCI Wholesale Customers** — 440 customers, 6 features (log-transformed + standardised). You will sweep K-Means across K=2…10, compute all four metrics, and then validate against the Channel proxy label. The metrics, not the visualisation, make the final call.


In [ ]:
# TODO: Implement this cell
#  (Setup)
#
# Steps:
# 1. Setup
# 2. Compute `IMG` using `Path()`
# 3. Load and preprocess
# 4. Fit the model -- call `log1p()`
# 5. Fit the model -- call `2D()`
# 6. Process data
#
# Hint:
#    IMG = Path(???)
#    scaler = StandardScaler(???)
#    df = pd.read_csv(???)
#    X_log = np.log1p(???)

## K Sweep: Computing All Three Internal Metrics

For each K from 2 to 10, compute silhouette, DBI, and CHI. Plot all three to find the optimal K — or at least the acceptable range.


In [ ]:
# TODO: Implement this cell
#  (K sweep: all three internal metrics)
#
# Steps:
# 1. K sweep: all three internal metrics
# 2. Fit the model -- call `append()`
# 3. Compute `results_df` using `DataFrame()`
#
# Hint:
#    km = KMeans(n_clusters=???, init=???)
#    results_df = pd.DataFrame(???)
#    km.fit(???)

In [ ]:
# TODO: Implement this cell
#  (Plot all three metrics vs K)
#
# Steps:
# 1. Plot all three metrics vs K
# 2. Plot results -- call `plot()`
# 3. Plot results -- call `suptitle()`
# 4. Compute `best_k_sil` using `argmax()`
#
# Hint:
#    axes = plt.subplots(???)

## Silhouette Subplot: Per-Segment Quality

The mean silhouette can hide a bad segment. Per-segment bar charts show which segments are well-defined and which are borderline.


In [ ]:
# TODO: Implement this cell
#  (Silhouette subplot for K=5)
#
# Steps:
# 1. Silhouette subplot for K=5
# 2. Compute `segment_names`
# 3. Plot results -- call `subplots()`
# 4. Call `sort()` to produce the result
# 5. Plot results -- call `tab10()`
# 6. Plot results -- call `axvline()`
# 7. Plot results -- call `tight_layout()`
# 8. Call `mean()` to produce the result
#
# Hint:
#    km5 = KMeans(n_clusters=???, init=???)
#    sil_mean = sil_vals.mean(???)
#    ax = plt.subplots(???)
#    color = plt.cm.tab10(???)

## Metric Disagreement: K=3 vs K=5

Silhouette says K=3. Business needs K=5. How to decide?


In [ ]:
# TODO: Implement this cell
#  (Metric disagreement analysis)
#
# Steps:
# 1. Metric disagreement analysis
# 2. Fit the model -- call `KMeans()`
# 3. Call `5()` to produce the result
#
# Hint:
#    km_k = KMeans(n_clusters=???, n_init=???)

## External Validation: ARI Against Channel Proxy

The `Channel` column (1=Hotel/Restaurant/Café, 2=Retail) was excluded from clustering. We can use it as proxy ground truth to validate that our clusters capture real structure.


In [ ]:
# TODO: Implement this cell
#  (ARI and NMI against Channel proxy)
#
# Steps:
# 1. ARI and NMI against Channel proxy
# 2. Compute `ari` using `adjusted_rand_score()`
# 3. Compute `cross` using `crosstab()`
#
# Hint:
#    cross = pd.crosstab(???)

## Bootstrap Stability (Constraint #3)

Do the segments survive resampling? For each of 100 bootstrap samples, re-cluster and check how consistently each customer is assigned to the same segment.


In [ ]:
# TODO: Implement this cell
#  (Bootstrap stability)
#
# Steps:
# 1. Bootstrap stability
# 2. Compute `n_boot` using `zeros()`
# 3. Fit the model -- call `RandomState()`
# 4. Compute `stability` using `array()`
# 5. Plot results -- call `subplots()`
# 6. Call `mean()` to produce the result
#
# Hint:
#    km_b = KMeans(n_clusters=???, n_init=???)
#    assignments = np.zeros(???)
#    rng = np.random.RandomState(???)
#    idx = rng.choice(???)

## Business Validation: Segment Profiles

The most important metric is: can the sales team act on these segments?


In [ ]:
# TODO: Implement this cell
#  (Segment profile summary)
#
# Steps:
# 1. Segment profile summary
# 2. Aggregate / merge data
# 3. Call `argmax()` to produce the result
#
# Hint:
#    centroids_log = scaler.inverse_transform(???)
#    n = mask.sum(???)
#    medians = np.median(???)

## Final SegmentAI Constraint Check


In [ ]:
# TODO: Implement this cell
#  (Final constraint verification)
#
# Steps:
# 1. Final constraint verification
# 2. Process data
# 3. Compute `constraints` using `O()`
# 4. Compute `all_pass`
# 5. Process data
#
# Hint:
#    mean_stab = stability.mean(???)

## What Can Go Wrong: Optimising Metrics vs Business Value


In [ ]:
# TODO: Implement this cell
#  (Demonstration: silhouette-optimal K vs business K)
#
# Steps:
# 1. Demonstration: silhouette-optimal K vs business K
# 2. Plot results -- call `3()`
# 3. Plot results -- call `suptitle()`
# 4. Process data
#
# Hint:
#    km_k = KMeans(n_clusters=???, n_init=???)
#    axes = plt.subplots(???)

## Exercises

1. **Silhouette at different preprocessing.** Compare silhouette for K=5 using: (a) raw 6D data, (b) log+scaled 6D, (c) PCA 2D, (d) PCA 4D. Which preprocessing gives the best silhouette?

2. **DBI decomposition.** For K=5, compute the DBI manually by finding the worst (most similar) pair for each segment. Which two segments are most similar? Should they be merged?

3. **Stability improvement.** If any customers have <70% bootstrap stability, examine their features. Are they boundary customers between two segments? Suggest a strategy to handle them (e.g., "uncertain" label, soft clustering).


In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Process data
#
# Hint:
#    # implement using the APIs described above

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Process data
#
# Hint:
#    # implement using the APIs described above

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Process data
#
# Hint:
#    # implement using the APIs described above